# Semana 6: Redes Neuronales para NLP y Embeddings Entrenables
## Notebook Conceptual (NB1) – Clasificador con Embeddings Entrenables

**Propósito:** Construir redes neuronales simples con PyTorch, introduciendo la capa de embedding y clasificadores basados en promedio de embeddings.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Construir un vocabulario y mapear palabras a índices.
- Implementar una capa de embedding en PyTorch (nn.Embedding).
- Crear un modelo que promedie los embeddings de las palabras y pase por una capa lineal.
- Entrenar para clasificación binaria (positivo/negativo) con entropía cruzada.
- Visualizar la evolución de la pérdida y accuracy.
- Comparar embeddings pre-entrenados (GloVe) fijos vs. entrenables.

---

## 0. Configuración Inicial

Importamos las librerías necesarias y configuramos PyTorch.

In [ ]:
# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import warnings
warnings.filterwarnings('ignore')

# Para embeddings pre-entrenados
import gensim.downloader as api

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Verificar PyTorch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

# Semilla
np.random.seed(42)
torch.manual_seed(42)

print("\nLibrerías importadas correctamente.")

---
## 1. Datos Dummy para Clasificación de Sentimiento

Creamos un pequeño conjunto de frases con sentimiento positivo (1) y negativo (0).

In [ ]:
# Frases de ejemplo
texts = [
    "me gusta este producto",
    "es terrible",
    "excelente calidad",
    "no me gusta nada",
    "muy bueno",
    "horrible experiencia",
    "lo recomiendo",
    "pesimo servicio",
    "fantastico",
    "no sirve"
]

# Etiquetas: 1 = positivo, 0 = negativo
labels = [1, 0, 1, 0, 1, 0, 1, 0, 1, 0]

print("=== DATOS ===")
for text, label in zip(texts, labels):
    sent = "POSITIVO" if label == 1 else "NEGATIVO"
    print(f"{sent:10} : {text}")

---
## 2. Construcción del Vocabulario y Codificación

### 2.1. Crear vocabulario con tokens especiales

In [ ]:
from collections import Counter

def build_vocab(texts, max_size=100):
    """Construye vocabulario a partir de textos."""
    # Tokenización simple por espacios
    all_tokens = []
    for text in texts:
        all_tokens.extend(text.lower().split())
    
    # Contar frecuencias
    word_counts = Counter(all_tokens)
    
    # Crear vocabulario con tokens especiales
    vocab = {'<PAD>': 0, '<UNK>': 1}
    
    # Añadir palabras más frecuentes
    for i, (word, _) in enumerate(word_counts.most_common(max_size)):
        vocab[word] = i + 2
    
    return vocab

vocab = build_vocab(texts, max_size=20)
print(f"Tamaño del vocabulario: {len(vocab)}")
print("Vocabulario:")
for word, idx in list(vocab.items())[:10]:
    print(f"  {word}: {idx}")

### 2.2. Convertir textos a índices con padding

In [ ]:
def encode_texts(texts, vocab, max_len=5):
    """Convierte textos a secuencias de índices con padding."""
    encoded = []
    for text in texts:
        tokens = text.lower().split()[:max_len]
        indices = [vocab.get(token, vocab['<UNK>']) for token in tokens]
        # Padding
        indices += [vocab['<PAD>']] * (max_len - len(indices))
        encoded.append(indices)
    return torch.tensor(encoded)

max_len = 4  # longitud máxima de secuencia
X = encode_texts(texts, vocab, max_len=max_len)
y = torch.tensor(labels)

print(f"Forma de X: {X.shape}")
print("Primeras 3 secuencias codificadas:")
for i in range(3):
    print(f"  {texts[i]}: {X[i].tolist()}")

---
## 3. Definición del Modelo

Creamos un clasificador que:
1. Convierte índices a embeddings
2. Promedia los embeddings de la secuencia
3. Pasa por una capa lineal con activación ReLU y dropout
4. Capa de salida para clasificación binaria

In [ ]:
class SentimentClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=50, hidden_dim=32, num_classes=2, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=vocab['<PAD>'])
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        # x: (batch_size, seq_len)
        embedded = self.embedding(x)  # (batch_size, seq_len, embed_dim)
        
        # Promedio de embeddings (ignorando padding)
        # Crear máscara para tokens que no son padding
        mask = (x != 0).unsqueeze(-1).float()  # (batch_size, seq_len, 1)
        
        # Suma de embeddings con máscara
        sum_embeddings = (embedded * mask).sum(dim=1)  # (batch_size, embed_dim)
        # Número de tokens reales por secuencia
        lengths = mask.sum(dim=1)  # (batch_size, 1)
        # Evitar división por cero
        pooled = sum_embeddings / (lengths + 1e-8)  # (batch_size, embed_dim)
        
        h = self.dropout(self.relu(self.fc1(pooled)))
        out = self.fc2(h)
        return out

# Instanciar modelo
model = SentimentClassifier(len(vocab), embed_dim=50, hidden_dim=32)
model.to(device)

# Contar parámetros
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total de parámetros: {total_params:,}")
print(f"Parámetros entrenables: {trainable_params:,}")

---
## 4. Entrenamiento del Modelo

### 4.1. Preparación de datos

In [ ]:
# Crear DataLoader
dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

# Función de pérdida y optimizador
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-5)

### 4.2. Bucle de entrenamiento

In [ ]:
epochs = 100
train_losses = []
train_accuracies = []

for epoch in range(epochs):
    epoch_loss = 0
    correct = 0
    total = 0
    
    for batch_X, batch_y in dataloader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        # Forward
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        
        # Backward
        loss.backward()
        optimizer.step()
        
        # Estadísticas
        epoch_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()
    
    epoch_loss /= len(dataloader)
    epoch_acc = correct / total
    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_acc)
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}")

### 4.3. Visualización de la evolución del entrenamiento

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses)
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Pérdida')
axes[0].set_title('Evolución de la pérdida')
axes[0].grid(True)

axes[1].plot(train_accuracies)
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Evolución del accuracy')
axes[1].grid(True)

plt.tight_layout()
plt.show()

---
## 5. Evaluación del Modelo

In [ ]:
model.eval()
with torch.no_grad():
    outputs = model(X.to(device))
    _, predicted = torch.max(outputs, 1)
    accuracy = (predicted.cpu() == y).float().mean()

print(f"Accuracy final en entrenamiento: {accuracy:.4f}")

# Mostrar predicciones
print("\nPredicciones:")
for text, true, pred in zip(texts, y, predicted.cpu()):
    true_sent = "POS" if true == 1 else "NEG"
    pred_sent = "POS" if pred == 1 else "NEG"
    correct = "✓" if true == pred else "✗"
    print(f"{correct} {text:25} -> Real: {true_sent}, Pred: {pred_sent}")

---
## 6. Comparación con Embeddings Pre-entrenados (GloVe)

### 6.1. Cargar embeddings GloVe

In [ ]:
# Descargar modelo GloVe de 50 dimensiones (si no está disponible)
print("Cargando embeddings GloVe (esto puede tomar un minuto)...")
try:
    glove_model = api.load('glove-twitter-25')  # versión pequeña para demo
    embed_dim_glove = 25
except:
    print("Usando alternativa: generando embeddings aleatorios")
    glove_model = None
    embed_dim_glove = 50

if glove_model:
    print("Embeddings GloVe cargados.")
    print(f"Dimensiones: {embed_dim_glove}")
    print(f"Vocabulario: {len(glove_model.key_to_index)} palabras")

### 6.2. Crear matriz de embeddings inicializada con GloVe

In [ ]:
def create_pretrained_embedding_matrix(vocab, glove_model, embed_dim):
    """Crea matriz de embeddings inicializada con GloVe."""
    vocab_size = len(vocab)
    embedding_matrix = np.zeros((vocab_size, embed_dim))
    
    for word, idx in vocab.items():
        if word in ['<PAD>', '<UNK>']:
            continue
        if glove_model and word in glove_model:
            embedding_matrix[idx] = glove_model[word]
        else:
            # Inicialización aleatoria para palabras no encontradas
            embedding_matrix[idx] = np.random.normal(0, 0.1, embed_dim)
    
    return torch.tensor(embedding_matrix, dtype=torch.float32)

if glove_model:
    pretrained_weights = create_pretrained_embedding_matrix(vocab, glove_model, embed_dim_glove)
    print(f"Matriz de embeddings creada: {pretrained_weights.shape}")

### 6.3. Definir modelos para comparación

1. **Modelo A**: Embeddings entrenables desde cero
2. **Modelo B**: Embeddings pre-entrenados congelados
3. **Modelo C**: Embeddings pre-entrenados y ajustables (fine-tuning)

In [ ]:
class SentimentClassifierPretrained(nn.Module):
    def __init__(self, embedding_weights, hidden_dim=32, num_classes=2, dropout=0.5, freeze=False):
        super().__init__()
        vocab_size, embed_dim = embedding_weights.shape
        self.embedding = nn.Embedding.from_pretrained(embedding_weights, freeze=freeze, padding_idx=0)
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        embedded = self.embedding(x)
        mask = (x != 0).unsqueeze(-1).float()
        sum_embeddings = (embedded * mask).sum(dim=1)
        lengths = mask.sum(dim=1)
        pooled = sum_embeddings / (lengths + 1e-8)
        h = self.dropout(self.relu(self.fc1(pooled)))
        out = self.fc2(h)
        return out

if glove_model:
    # Modelo con embeddings congelados
    model_frozen = SentimentClassifierPretrained(pretrained_weights, freeze=True).to(device)
    
    # Modelo con embeddings ajustables (fine-tuning)
    model_tunable = SentimentClassifierPretrained(pretrained_weights, freeze=False).to(device)
    
    print("Modelos con embeddings pre-entrenados creados.")

### 6.4. Entrenamiento comparativo (simplificado)

Para este ejemplo, entrenaremos los tres modelos y compararemos el accuracy final.

In [ ]:
def train_model(model, dataloader, epochs=50):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    
    for epoch in range(epochs):
        for batch_X, batch_y in dataloader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
    
    # Evaluar
    model.eval()
    with torch.no_grad():
        outputs = model(X.to(device))
        _, predicted = torch.max(outputs, 1)
        accuracy = (predicted.cpu() == y).float().mean()
    return accuracy.item()

print("Entrenando modelos comparativos...")

# Modelo desde cero (ya entrenado)
acc_scratch = accuracy.item()

if glove_model:
    # Modelo con embeddings congelados
    acc_frozen = train_model(model_frozen, dataloader, epochs=50)
    
    # Modelo con embeddings ajustables
    acc_tunable = train_model(model_tunable, dataloader, epochs=50)
    
    print("\n=== COMPARACIÓN ===")
    print(f"Desde cero (entrenables): {acc_scratch:.4f}")
    print(f"Pre-entrenados congelados: {acc_frozen:.4f}")
    print(f"Pre-entrenados ajustables: {acc_tunable:.4f}")
else:
    print(f"Desde cero (entrenables): {acc_scratch:.4f}")

### Análisis de la comparación:

- **Embeddings desde cero**: Pueden funcionar bien si hay suficientes datos, pero requieren más épocas.
- **Embeddings congelados**: Útiles cuando se tienen pocos datos, ya que aprovechan el conocimiento de GloVe.
- **Embeddings ajustables**: Combinan lo mejor de ambos mundos, permitiendo adaptar los embeddings a la tarea específica.

En este ejemplo con datos muy pequeños, es posible que los embeddings desde cero ya hayan memorizado los datos. En un caso real con más datos, los pre-entrenados suelen dar mejores resultados.

---
## 7. Ejercicio para el Estudiante

1. Añade más frases al dataset y observa cómo cambia el rendimiento.
2. Experimenta con diferentes dimensiones de embedding (10, 50, 100).
3. Prueba a cambiar la arquitectura: añade una capa oculta adicional.
4. Implementa max-pooling en lugar de average-pooling y compara.
5. Si tienes acceso a un dataset real (como IMDb), prueba este modelo en él.

In [ ]:
# Espacio para el estudiante
# ...

---
## 8. Conclusiones

En este notebook hemos construido nuestro primer clasificador neuronal para NLP:

✔️ **Vocabulario y codificación**: Aprendimos a convertir texto a índices y manejar padding.

✔️ **Capa de embedding**: Implementamos `nn.Embedding` y entendimos cómo se entrena.

✔️ **Arquitectura**: Modelo que promedia embeddings y pasa por MLP.

✔️ **Entrenamiento**: Bucle completo con visualización de pérdida y accuracy.

✔️ **Embeddings pre-entrenados**: Comparamos GloVe fijo vs entrenable.

**Lección clave**: Las redes neuronales con embeddings entrenables son una herramienta poderosa y flexible para NLP. Sirven como base para arquitecturas más complejas como RNNs y Transformers.

En el próximo notebook (NB2) aplicaremos este modelo a un dataset real de IMDb.

---
**Fin del Notebook Conceptual - Semana 6 (NLP)**